# T4 — Italy is frozen: but for whom?

**Research question (final, locked):** *Italian real wages have been flat in aggregate since the 1990s (T1). When the freeze is broken down by age band, did all groups stagnate equally, or did some lose more than others?*

**Why this question.** T1 establishes the macro fact (Italy is flat against European peers); T2 the geographic dimension; T3 the firm-size dimension. T4 closes the loop with the **generational dimension** — same country, same methodology, different age bands.

**Approach.** ISTAT RACLI (administrative wage register, all firm sizes including micro) is the single source: median gross hourly wage of Italian dependent workers, 2014–2023, by age band (15–29, 30–49, 50+). All values are deflated to **2023 euros** with the Eurostat HICP. No splicing across sources is performed: we accept the shorter (10-year) window in exchange for full methodological consistency.

**Audience.** Italian readers of long-form economic journalism (Il Post, Domani, Il Sole 24 Ore Plus). The pictogram answers the question in five seconds.

---

## What the data say (preview)

The naïve hypothesis going in was *"Italy is frozen — but the freeze hits the young hardest"* (the *asymmetric freeze* idea). The data do **not** support it. Between 2014 and 2023, real hourly wages declined for *every* age band. The 50+ band — often presumed sheltered by tenure and collective bargaining — actually lost the most in real percentage terms. The headline of T4 is therefore not *young vs old* but *uniform decline, with a counter-intuitive twist at the senior end*.

This finding is itself part of the story.

---

## Notebook structure

1. **Setup** — imports, palette, paths.
2. **Load ISTAT RACLI** — 2014–2023, three age bands.
3. **HICP deflator** — Eurostat HICP (Italy), reproducible from API.
4. **Build the real-wage frame** — long format, tidy.
5. **Headline finding** — % change by band, 2014→2023.
6. **Pictogram (static, matplotlib)** — newspaper-style coin stacks: 15–29 vs 50+ side by side, three snapshot years.
7. **Pictogram (interactive, Plotly)** — three age bands, with a toggle that adds/removes the middle band 30–49.
8. **Save outputs.**


## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
import json
import ssl
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_DIR = Path("data/task_4")
VIZ_DIR = Path("site/viz")
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Newspaper palette (locked in 02_viz_plan.md)
ACCENT = "#c44536"      # editorial red — used for the 15-29 band
INK    = "#1a1a1a"      # ink black — used for the 50+ band (the "stable" anchor that turns out not to be)
MID    = "#8a8a8a"      # mid grey — used for the 30-49 band in the interactive toggle
GRAY   = "#666666"      # secondary text
BG     = "#f8f5ec"
GRID   = "#e0e0e0"


## 2. Load ISTAT RACLI (2014–2023, three age bands)

ISTAT *Rilevazione su Costo Lavoro e Retribuzioni* (DCSC_RACLI). Median gross hourly wage of Italian dependent workers, by age × firm-size class × NACE sector, annual 2014–2023. Critically, this dataset **includes micro-firms (0–9 employees)** — the segment Eurostat earn_ses_pub2a excludes by design. We use the headline series: NACE total (`0010`), all firm sizes (`TOTAL`), and the three age bands published by ISTAT — 15–29, 30–49, 50+.

In [2]:
src_istat = DATA_DIR / "Classe di età (IT1,533_957_DF_DCSC_RACLI_1,1.0).csv"
istat_raw = pd.read_csv(src_istat, encoding='utf-8-sig', engine='python', on_bad_lines='skip')

istat = (istat_raw[['AGE', 'EMPLOYESS_CLASS', 'ECON_ACTIVITY_NACE_2007',
                    'TIME_PERIOD', 'Osservazione']]
         .rename(columns={
             'AGE': 'age',
             'EMPLOYESS_CLASS': 'sizeclass',
             'ECON_ACTIVITY_NACE_2007': 'nace',
             'TIME_PERIOD': 'year',
             'Osservazione': 'wage_eur_h_nom',
         }))
istat['year'] = istat['year'].astype(int)
istat['wage_eur_h_nom'] = pd.to_numeric(istat['wage_eur_h_nom'], errors='coerce')
istat = istat.dropna(subset=['wage_eur_h_nom'])

# Headline subset: NACE total, all firm sizes, three age bands.
AGE_BANDS = ['Y15-29', 'Y30-49', 'Y_GE50']
ist_h = (istat[(istat.nace == '0010')
               & (istat.sizeclass == 'TOTAL')
               & (istat.age.isin(AGE_BANDS))]
         [['year', 'age', 'wage_eur_h_nom']]
         .sort_values(['age', 'year'])
         .reset_index(drop=True))

print("ISTAT RACLI — Italy, dependent workers, all firm sizes, hourly nominal wage (€):")
print(ist_h.pivot(index='year', columns='age', values='wage_eur_h_nom').round(2).to_string())


ISTAT RACLI — Italy, dependent workers, all firm sizes, hourly nominal wage (€):
age   Y15-29  Y30-49  Y_GE50
year                        
2014    9.76   11.34   12.39
2015    9.90   11.45   12.52
2016    9.92   11.46   12.48
2017   10.03   11.49   12.46
2018   10.06   11.51   12.40
2019   10.15   11.65   12.53
2020   10.35   11.92   12.82
2021   10.37   11.93   12.84
2022   10.50   12.00   12.85
2023   10.70   12.24   13.08


## 3. HICP deflator (Italy, Eurostat, base 2015 = 100)

Brings every nominal wage into **2023 €** so they are directly comparable in purchasing-power terms. Live API call from the Eurostat REST endpoint — fully reproducible from a clean clone of the repo.

In [3]:
# Eurostat HICP via the dissemination API.  The certificate-verification bypass
# below is a workaround for a corporate certificate chain on the developer
# machine that intermittently fails to resolve Eurostat's CA. Re-running this
# notebook on a network without that constraint can use the system default
# context (drop the three lines below and pass no `context=` to urlopen).
url = ("https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
       "prc_hicp_aind?geo=IT&coicop=CP00&unit=INX_A_AVG&format=JSON")
ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
with urllib.request.urlopen(url, context=ctx, timeout=20) as r:
    j = json.loads(r.read())
yi = j['dimension']['time']['category']['index']
vals = j['value']
hicp = (pd.DataFrame([(int(y), vals.get(str(i)))
                       for y, i in yi.items()],
                      columns=['year', 'hicp_2015_100'])
          .dropna()
          .reset_index(drop=True))

hicp_2023 = hicp.loc[hicp.year == 2023, 'hicp_2015_100'].iloc[0]
hicp['def_2023'] = hicp_2023 / hicp['hicp_2015_100']
hicp.to_csv(DATA_DIR / 'hicp_italy_eurostat_annual.csv', index=False)

print("HICP series 2014-2024 (base 2015 = 100):")
print(hicp[hicp.year.between(2014, 2024)][['year', 'hicp_2015_100', 'def_2023']]
      .round(3).to_string(index=False))

infl_14_23 = (hicp_2023 / hicp.loc[hicp.year == 2014, 'hicp_2015_100'].iloc[0] - 1) * 100
print(f"\nCumulative inflation 2014 → 2023: {infl_14_23:+.1f}%")


HICP series 2014-2024 (base 2015 = 100):
 year  hicp_2015_100  def_2023
 2014           99.9     1.210
 2015          100.0     1.209
 2016           99.9     1.210
 2017          101.3     1.193
 2018          102.5     1.180
 2019          103.2     1.172
 2020          103.0     1.174
 2021          105.0     1.151
 2022          114.2     1.059
 2023          120.9     1.000
 2024          122.3     0.989

Cumulative inflation 2014 → 2023: +21.0%


## 4. Build the real-wage frame

A single tidy long-format frame: median real hourly wage of Italian dependent workers, 2014–2023, by age band, in 2023 €.

In [4]:
real_wages = (ist_h
               .merge(hicp[['year', 'def_2023']], on='year')
               .assign(wage_eur_h_real_2023=lambda d: d.wage_eur_h_nom * d.def_2023)
               [['year', 'age', 'wage_eur_h_nom', 'def_2023', 'wage_eur_h_real_2023']]
               .sort_values(['age', 'year'])
               .reset_index(drop=True))

real_wages.to_csv(DATA_DIR / 'italy_real_wages_by_age_2014_2023.csv', index=False)

print("Real hourly wages (in 2023 €) — Italy, by age band:\n")
print(real_wages.pivot(index='year', columns='age', values='wage_eur_h_real_2023')
        .round(2).to_string())


Real hourly wages (in 2023 €) — Italy, by age band:

age   Y15-29  Y30-49  Y_GE50
year                        
2014   11.81   13.72   14.99
2015   11.97   13.84   15.14
2016   12.01   13.87   15.10
2017   11.97   13.71   14.87
2018   11.87   13.58   14.63
2019   11.89   13.65   14.68
2020   12.15   13.99   15.05
2021   11.94   13.74   14.78
2022   11.12   12.70   13.60
2023   10.70   12.24   13.08


## 5. Headline finding

In [5]:
wide = real_wages.pivot(index='year', columns='age', values='wage_eur_h_real_2023')

print("Real hourly wage change 2014 → 2023, by age band (in 2023 €):\n")
for band in AGE_BANDS:
    val_14 = wide.loc[2014, band]
    val_23 = wide.loc[2023, band]
    pct = (val_23 / val_14 - 1) * 100
    eur = val_23 - val_14
    label = {'Y15-29': '15–29 (young)',
             'Y30-49': '30–49 (mid-career)',
             'Y_GE50': '50+ (senior)'}[band]
    print(f"  {label:25s}  €{val_14:5.2f}  →  €{val_23:5.2f}  "
          f"({pct:+5.1f}% real, {eur:+5.2f} €/h)")

print()
print("HEADLINE: All three age bands lost real-wage ground over the decade 2014–2023.")
print("          The 50+ band — supposedly sheltered — lost the most (-12.8%),")
print("          followed by 30–49 (-10.8%) and 15–29 (-9.4%).")
print("          The decline is concentrated post-2021 (cumulative HICP +15% in 2021–2023,")
print("          not absorbed by collective-bargaining adjustments).")


Real hourly wage change 2014 → 2023, by age band (in 2023 €):

  15–29 (young)              €11.81  →  €10.70  ( -9.4% real, -1.11 €/h)
  30–49 (mid-career)         €13.72  →  €12.24  (-10.8% real, -1.48 €/h)
  50+ (senior)               €14.99  →  €13.08  (-12.8% real, -1.91 €/h)

HEADLINE: All three age bands lost real-wage ground over the decade 2014–2023.
          The 50+ band — supposedly sheltered — lost the most (-12.8%),
          followed by 30–49 (-10.8%) and 15–29 (-9.4%).
          The decline is concentrated post-2021 (cumulative HICP +15% in 2021–2023,
          not absorbed by collective-bargaining adjustments).


**Methodological note to flag in the chart caption:**

We are showing the median, not the entire distribution. The same RACLI dataset publishes deciles; for a distributional analysis (out of scope here) those would be the right primitive.


## 6. Pictogram — interactive (Plotly)

Same data, three age bands (15-29 / 30-49 / 50+) plus a **year-by-year animation** that reveals the snapshots in chronological order:

1. **Frame 1 — Baseline 2014.** Three stacks visible (one per age band). The starting reference.
2. **Frame 2 — + 2021.** A second column appears next to 2014. Real wages have barely moved in seven years — the new column looks almost identical to the first. Italy looks frozen but stable.
3. **Frame 3 — + 2023.** The third column appears, visibly shorter than the previous two. Decline annotations (Δ% vs 2014) pop up: −9.4% / −10.8% / −12.8%. The post-COVID inflation cliff is now visible at a glance.

The animation order mirrors the headline finding: *"stable for seven years, then a cliff in two."*

In [6]:
# ---------------------------------------------------------------
#   T4 PICTOGRAM, build script
# ---------------------------------------------------------------
# Builds an interactive Plotly figure that stacks one euro-coin per
# euro of median real hourly wage, repeated for each (year, age band)
# combination, plus an animation across three featured years
# (2014, 2021, 2023). The chart structure is:
#
#   X axis = three "year" columns spaced by OUTER_DX, with three age
#            bands (15-29, 30-49, 50+) inside each column spaced by
#            INNER_DX.
#   Y axis = stacked euro coins, one marker per integer euro plus a
#            half-coin marker for the residual.
#   Annotations = year headers above each column and a percent-change
#                 callout above the 2021 and 2023 stacks (vs 2014).
#
# Trace registry (used by the band-filter dropdown and the animation
# frame builder):
#   * `year_trace_idx[y]`         indices of all traces that belong
#                                 to year column `y`
#   * `band_trace_idx[b]`         indices of all traces for band `b`
#   * `year_header_idx`           indices of the year-header label
#                                 traces (band-agnostic)
#   * `decline_idx_by_band[b]`    indices of the percent-change
#                                 callouts for band `b`
#
# The animation emits three frames:
#   Frame 1: only the 2014 column.
#   Frame 2: 2014 + 2021 columns.
#   Frame 3: 2014 + 2021 + 2023 columns plus the percent-change
#            callouts.
# ---------------------------------------------------------------
import plotly.graph_objects as go

YEARS_FEAT = [2014, 2021, 2023]
BANDS_3    = ['Y15-29', 'Y30-49', 'Y_GE50']
COLORS_3   = {'Y15-29': ACCENT, 'Y30-49': MID, 'Y_GE50': INK}
LABEL_3    = {'Y15-29': '15–29', 'Y30-49': '30–49', 'Y_GE50': '50+'}

INNER_DX    = 1.2
OUTER_DX    = 5.0
COIN_DY     = 1.0
MARKER_SIZE = 22

wages_3 = wide.loc[YEARS_FEAT, BANDS_3]

fig = go.Figure()

# Track which traces belong to which (year, band, kind) for filter logic
year_trace_idx = {y: [] for y in YEARS_FEAT}
band_trace_idx = {b: [] for b in BANDS_3}
year_header_idx = []        # year header labels (band-agnostic)
decline_idx_by_band = {}    # decline annotation per band

for i, year in enumerate(YEARS_FEAT):
    cx = i * OUTER_DX
    for j, band in enumerate(BANDS_3):
        x = cx + (j - 1) * INNER_DX
        val = wages_3.loc[year, band]
        col = COLORS_3[band]
        n_coins = int(np.floor(val))
        residue = val - n_coins
        delta_2014 = (val / wages_3.loc[2014, band] - 1) * 100
        hover = (f"<b>{year}</b> &nbsp;·&nbsp; age {LABEL_3[band]}<br>"
                 f"Real wage: <b>€{val:.2f}/h</b>"
                 + (f"<br>Δ vs 2014: <b>{delta_2014:+.1f}%</b>" if year != 2014 else ""))
        # Coin trace — € on top integer coin (when no residue)
        if n_coins > 0:
            coin_texts = [''] * n_coins
            if residue <= 0.05:
                coin_texts[-1] = '€'
            fig.add_trace(go.Scatter(
                x=[x] * n_coins,
                y=[k * COIN_DY for k in range(n_coins)],
                mode='markers+text',
                marker=dict(size=MARKER_SIZE, color=col, line=dict(color='white', width=1.2)),
                text=coin_texts,
                textfont=dict(size=10, color='white', family='serif'),
                textposition='middle center',
                hoverinfo='text', hovertext=[hover] * n_coins,
                showlegend=False,
            ))
            ti = len(fig.data) - 1
            year_trace_idx[year].append(ti)
            band_trace_idx[band].append(ti)
        if residue > 0.05:
            fig.add_trace(go.Scatter(
                x=[x], y=[n_coins * COIN_DY],
                mode='markers+text',
                marker=dict(size=MARKER_SIZE, color=col,
                            opacity=0.30 + 0.65 * residue,
                            line=dict(color='white', width=1.2)),
                text=['€'],
                textfont=dict(size=10, color='white', family='serif'),
                textposition='middle center',
                hoverinfo='text', hovertext=[hover],
                showlegend=False,
            ))
            ti = len(fig.data) - 1
            year_trace_idx[year].append(ti)
            band_trace_idx[band].append(ti)
            top_y = (n_coins + 1) * COIN_DY
        else:
            top_y = n_coins * COIN_DY
        # Value label above stack
        fig.add_trace(go.Scatter(
            x=[x], y=[top_y + 0.5],
            mode='text', text=[f"<b>€{val:.2f}</b>"],
            textfont=dict(size=11, color=col, family='Playfair Display, serif'),
            textposition='top center',
            hoverinfo='skip', showlegend=False,
        ))
        ti = len(fig.data) - 1
        year_trace_idx[year].append(ti)
        band_trace_idx[band].append(ti)
        # Band label below stack
        fig.add_trace(go.Scatter(
            x=[x], y=[-0.7],
            mode='text', text=[LABEL_3[band]],
            textfont=dict(size=11, color=col),
            textposition='bottom center',
            hoverinfo='skip', showlegend=False,
        ))
        ti = len(fig.data) - 1
        year_trace_idx[year].append(ti)
        band_trace_idx[band].append(ti)

# Year header labels (big year, under each group) — band-agnostic, appear with their year
for i, year in enumerate(YEARS_FEAT):
    fig.add_trace(go.Scatter(
        x=[i * OUTER_DX], y=[-2.2],
        mode='text', text=[f"<b>{year}</b>"],
        textfont=dict(size=22, color=INK, family='Playfair Display, serif'),
        textposition='bottom center',
        hoverinfo='skip', showlegend=False,
    ))
    ti = len(fig.data) - 1
    year_trace_idx[year].append(ti)
    year_header_idx.append(ti)

# Decline annotations (Δ% vs 2014) — only on frame 3
for j, band in enumerate(BANDS_3):
    val_14 = wages_3.loc[2014, band]
    val_23 = wages_3.loc[2023, band]
    pct = (val_23 / val_14 - 1) * 100
    col = COLORS_3[band]
    x_23 = (len(YEARS_FEAT) - 1) * OUTER_DX + (j - 1) * INNER_DX
    n_23 = int(np.floor(val_23))
    res_23 = val_23 - n_23
    top_23 = (n_23 + (1 if res_23 > 0.05 else 0)) * COIN_DY
    fig.add_trace(go.Scatter(
        x=[x_23], y=[top_23 + 1.7],
        mode='text', text=[f"<b>{pct:+.1f}%</b><br><span style='font-size:9px'>vs 2014</span>"],
        textfont=dict(size=13, color=col, family='Playfair Display, serif'),
        textposition='top center',
        hoverinfo='skip', showlegend=False,
    ))
    decline_idx_by_band[band] = len(fig.data) - 1

n_traces = len(fig.data)
all_decline = list(decline_idx_by_band.values())

# === Year-based animation frames (cumulative reveal) ====================
def vis_year_progression(visible_years, include_decline):
    visible = set()
    for y in visible_years:
        visible.update(year_trace_idx[y])
    if include_decline:
        visible.update(all_decline)
    return [(i in visible) for i in range(n_traces)]

frame_steps = [
    ('1. 2014 baseline',                   [2014],                False),
    ('2. + 2021 — still flat',             [2014, 2021],          False),
    ('3. + 2023 — the post-COVID cliff',   [2014, 2021, 2023],    True),
]

fig.frames = [
    go.Frame(name=label,
             data=[dict(visible=v) for v in vis_year_progression(years, decline)],
             traces=list(range(n_traces)))
    for label, years, decline in frame_steps
]

# === Band filter visibility states (restyle) ============================
def vis_band_filter(target_band):
    """Show only one band across all years, plus year headers and that band's decline."""
    visible = set(band_trace_idx[target_band])
    visible.update(year_header_idx)
    visible.add(decline_idx_by_band[target_band])
    return [(i in visible) for i in range(n_traces)]

vis_all = [True] * n_traces  # full state — same as frame 3 fully revealed

# Initial state: only 2014 visible (matches frame 1)
for ti, vis in enumerate(vis_year_progression([2014], False)):
    fig.data[ti].visible = vis

max_y = float(wages_3.values.max()) + 4.5

fig.update_layout(
    title=dict(
        text=("<b>Italy is frozen — but for everyone?</b><br>"
              "<span style='font-size:13px;color:#666'><i>Real hourly wages of "
              "Italian dependent workers, 2014–2023 (in 2023 €).</i></span>"),
        x=0.05, xanchor='left', y=0.97, yanchor='top',
        font=dict(size=22, color=INK, family='Playfair Display, serif')),
    plot_bgcolor=BG, paper_bgcolor=BG,
    width=900, height=820,
    margin=dict(t=170, b=120, l=40, r=40),
    xaxis=dict(visible=False,
               range=[-OUTER_DX/2 + 0.3, (len(YEARS_FEAT)-1) * OUTER_DX + OUTER_DX/2 - 0.3],
               showgrid=False, zeroline=False),
    yaxis=dict(visible=False, range=[-3.6, max_y],
               showgrid=False, zeroline=False),
    hovermode='closest',
    hoverlabel=dict(bgcolor='#1a3a5c', font_color='white', font_size=12,
                    font_family='Inter, sans-serif'),
    updatemenus=[
        # 1) Band filter dropdown (top left, below title)
        dict(
            type='dropdown', direction='down',
            x=0.05, y=1.04, xanchor='left', yanchor='bottom',
            showactive=True,
            buttons=[
                dict(label='All age bands',
                     method='restyle',
                     args=[{'visible': vis_all}]),
                dict(label='Only 15–29',
                     method='restyle',
                     args=[{'visible': vis_band_filter('Y15-29')}]),
                dict(label='Only 30–49',
                     method='restyle',
                     args=[{'visible': vis_band_filter('Y30-49')}]),
                dict(label='Only 50+',
                     method='restyle',
                     args=[{'visible': vis_band_filter('Y_GE50')}]),
            ],
        ),
        # 2) Year animation buttons (top right, below title)
        dict(
            type='buttons', direction='right',
            x=0.99, y=1.04, xanchor='right', yanchor='bottom',
            showactive=False,
            buttons=[
                dict(label='▶ Watch the decade',
                     method='animate',
                     args=[None, {
                         'frame': {'duration': 1700, 'redraw': True},
                         'transition': {'duration': 600},
                         'fromcurrent': True,
                         'mode': 'immediate',
                     }]),
                dict(label='⏮ Restart',
                     method='animate',
                     args=[[frame_steps[0][0]], {
                         'frame': {'duration': 0, 'redraw': True},
                         'mode': 'immediate',
                     }]),
                dict(label='⏭ Skip to end',
                     method='animate',
                     args=[[frame_steps[-1][0]], {
                         'frame': {'duration': 0, 'redraw': True},
                         'mode': 'immediate',
                     }]),
            ],
        ),
    ],
    sliders=[dict(
        active=0,
        x=0.06, y=-0.05, len=0.88,
        currentvalue=dict(prefix='', font=dict(color=INK, size=12, family='serif')),
        pad=dict(t=20),
        steps=[dict(label=label,
                    method='animate',
                    args=[[label], {
                        'frame': {'duration': 700, 'redraw': True},
                        'mode': 'immediate',
                    }])
               for label, _, _ in frame_steps],
    )],
)
fig.add_annotation(
    x=0, y=-0.15, xref='paper', yref='paper', xanchor='left',
    text=("<i>One coin = €1 of gross hourly wage in real 2023 euros (HICP-deflated). "
          "Source: ISTAT RACLI 2014–2023, dependent workers, all firm sizes. "
          
          "Tip: pressing ▶ resets the band filter to All.</i>"),
    showarrow=False,
    font=dict(size=10, color=GRAY, family='EB Garamond, serif'),
)

out_html = VIZ_DIR / 't4_pictogram.html'
fig.write_html(out_html, include_plotlyjs='cdn', full_html=True)
print(f"Saved: {out_html}")
fig.show()

# --- centre the chart div within its iframe (post-processing) ---
_centering_css = (
    "<style>\n"
    "  html, body { margin: 0; padding: 0; }\n"
    "  .plotly-graph-div { margin: 0 auto !important; }\n"
    "</style>\n"
)
_html = open(str(out_html)).read()
if "plotly-graph-div { margin: 0 auto" not in _html:
    _html = _html.replace("</head>", "</head>\n" + _centering_css)
    open(str(out_html), "w").write(_html)



Saved: site/viz/t4_pictogram.html


## 7. Outputs

- `data/task_4/italy_real_wages_by_age_2014_2023.csv` — tidy real-wage frame, three age bands.
- `data/task_4/hicp_italy_eurostat_annual.csv` — HICP deflator (refreshable from API on every run).
- `site/viz/t4_pictogram.html` — interactive pictogram with mid-band toggle.

Documentation: `data/task_4/SOURCES.md`.
